In [ ]:
import numpy as np 
from scipy.linalg import expm

In [ ]:
hadamard = np.array([[1.,1.],[1.,-1.]], dtype=complex)/np.sqrt(2)

In [ ]:
def unitary(t, alpha):
    return np.diag(np.exp(-1j*t*alpha*np.array([1, -1])))

In [ ]:
state = np.array([[1.],[0.]], dtype=complex)
state

In [ ]:
alpha = 0.3
t = 1.

u = unitary(t, alpha)
final_state = hadamard @ (u @ (hadamard @ state))

probs = np.abs(final_state)**2
probs

In [ ]:
alpha_est = np.arccos(2*probs[0, 0] - 1)/(2*t)
alpha_est

In [ ]:
import pennylane as qml
import numpy as np

In [ ]:
# hamiltonian = - (param/2) Z
sampling_time = 1.
param = 0.3

n_qubits = 2
dev = qml.device("default.qubit", n_qubits)

@qml.qnode(dev)
def circuit():
    qml.Hadamard(0)
    qml.RZ(param, 0)
    qml.Hadamard(0)
    # ansatz.make(x)
    # qml.adjoint(ansatz.make)(y)
    # return qml.expval(qml.Projector([0] * n_wires, wires=range(n_wires)))
    return qml.probs()

In [ ]:
circuit()

probs = circuit()[0]

param_est = np.arccos(2*probs - 1)/sampling_time
param_est

In [28]:
import torch
import hornbill
import numpy as np

import matplotlib.pyplot as plt

In [35]:
gamma = 1.0
time = 1.0

state = hornbill.StateVector(2)
print(state.probabilities())
state.apply(hornbill.gates.H, [0])
state.apply(hornbill.gates.CNOT)
print(state.probabilities())

state.apply(hornbill.gates.H, [0])
state.apply(hornbill.gates.H, [1])
state.apply(hornbill.gates.rz(gamma*time), [0])
state.apply(hornbill.gates.rz(gamma*time), [1])
state.apply(hornbill.gates.H, [0])
state.apply(hornbill.gates.H, [1])

#state.apply(hornbill.gates.rz)

print(state.probabilities())


tensor([1., 0., 0., 0.], dtype=torch.float64)
tensor([0.5000, 0.0000, 0.0000, 0.5000], dtype=torch.float64)
tensor([0.1460, 0.3540, 0.3540, 0.1460], dtype=torch.float64)


In [25]:
# H = -(gamma/2) \sum_i Z_i
# \sum_i Z_i |s> = (Z_1+Z_2+...)|s> = ((-1)^{s1}+(-1)^{s2}+...)|s>
# = (n0 - n1)|s>
#n0+n1=n, = n-2n1

zsum_diag = torch.tensor([2-2*np.binary_repr(i, 2).count('1') for i in range(2**2)], dtype=torch.complex128)
ham_diag = -(gamma/2) * zsum_diag
unitary_diag = torch.exp(-1j*ham_diag*time)

h_all = torch.kron(hornbill.gates.H, hornbill.gates.H)

state = hornbill.zero_state(2)
state = h_all @ state
state = unitary_diag * state
state = h_all @ state
print(state.real**2+state.imag**2)

tensor([0.5931, 0.1770, 0.1770, 0.0528], dtype=torch.float64)
